In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install japanize-matplotlib

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error
import itertools
import math
import re
import lightgbm as lgb
import japanize_matplotlib
from sklearn.metrics import mean_squared_log_error
import numpy as np

In [ ]:
idk = pd.IndexSlice

In [ ]:
train =pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv")
test=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv")
oil=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv")
stores=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
transactions=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv")
holidays=pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv")

In [ ]:
train['test'] = 0
test['test'] = 1

data = pd.concat([train, test], axis=0)

In [ ]:
data = data.merge(holidays, on='date', how='left')
data= data.merge(stores, on='store_nbr', how='left')
data= data.merge(oil, on='date', how='left')
data= data.merge(transactions, on=['date', 'store_nbr'], how='left')
data = data.set_index(['store_nbr', 'date', 'family'])
data = data.drop(index='2013-01-01', level=1)
data

In [ ]:
data_ = data.copy().reset_index()

train = data_[data_['test'] == 0]
test = data_[data_['test'] == 1]

train['date'] = pd.to_datetime(train["date"])
train['day_of_week'] = train['date'].dt.day_of_week
train['day_of_year'] = train['date'].dt.dayofyear
train['day_of_month'] = train['date'].dt.day
train['month'] = train['date'].dt.month
train['quarter'] = train['date'].dt.quarter
train['year'] = train['date'].dt.year

In [ ]:
# trainデータの日付をdatetime型に変換
train['date'] = pd.to_datetime(train['date'])
train['weekday'] = train['date'].dt.day_name()

# 曜日ごとの平均売上を計算
weekday_sales = train.groupby('weekday')['sales'].mean().reindex([
    'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
])

# グラフの描画
plt.figure(figsize=(10, 6))
sns.barplot(x=weekday_sales.index, y=weekday_sales.values)
plt.title('曜日ごとの平均売上', fontsize=16)
plt.ylabel('平均売上')
plt.xlabel('曜日')
plt.savefig('週間売り上げ') # グラフをファイルとして保存
plt.show()

In [ ]:
# プロモーションの有無で平均売上を計算
promo_avg_sales = train.groupby(train['onpromotion'] > 0)['sales'].mean().reset_index()
promo_avg_sales['onpromotion'] = promo_avg_sales['onpromotion'].apply(lambda x: 'プロモーションあり' if x else 'プロモーションなし')


plt.figure(figsize=(8, 6))
sns.barplot(x='onpromotion', y='sales', data=promo_avg_sales)

plt.title('プロモーションの有無による平均売上の比較', fontsize=16)
plt.ylabel('平均売上')
plt.xlabel('')

plt.savefig('promotion_barchart.png')
plt.show()

In [ ]:
grouping_columns = ['year', 'quarter', 'month', 'day_of_week', 'day_of_year', 'day_of_month']

for ind, column in enumerate(grouping_columns):
    grouped_data = train.groupby(column)['sales'].sum()
    grouped_data = pd.DataFrame(grouped_data).reset_index()

In [ ]:
data_ = data.copy().reset_index()

# 店舗×商品カテゴリごとにグループ化
grouped_data = data_.groupby(['store_nbr', 'family'])

# --- 売上(sales)に関する移動特徴量 ---
# より多くの期間の移動平均を追加
data_['sales_rolling_mean_14'] = grouped_data['sales'].transform(lambda x: x.shift(1).rolling(14).mean())
data_['sales_rolling_mean_28'] = grouped_data['sales'].transform(lambda x: x.shift(1).rolling(28).mean())

# 売上のばらつきを示す「移動標準偏差」を追加
data_['sales_rolling_std_7'] = grouped_data['sales'].transform(lambda x: x.shift(1).rolling(7).std())
data_['sales_rolling_std_14'] = grouped_data['sales'].transform(lambda x: x.shift(1).rolling(14).std())
data_['sales_rolling_std_28'] = grouped_data['sales'].transform(lambda x: x.shift(1).rolling(28).std())


# --- 石油価格(dcoilwtico)に関する移動特徴量 ---
# 石油価格は店舗や商品によらないため、全体で計算
data_['oil_rolling_mean_7'] = data_['dcoilwtico'].transform(lambda x: x.shift(1).rolling(7).mean())
data_['oil_rolling_mean_14'] = data_['dcoilwtico'].transform(lambda x: x.shift(1).rolling(14).mean())
data_['oil_rolling_mean_28'] = data_['dcoilwtico'].transform(lambda x: x.shift(1).rolling(28).mean())


# 7日間の移動平均は、セル9で可視化に使ったので、ここで再作成します
data_['sales_rolling_mean_7'] = grouped_data['sales'].transform(lambda x: x.shift(1).rolling(7).mean())

# 最初の数日は計算できないのでNaN（欠損値）になる。0で埋めておく。
cols_to_fill = [col for col in data_.columns if 'rolling' in col]
data_[cols_to_fill] = data_[cols_to_fill].fillna(0)

In [ ]:
alphas = [0.95, 0.8, 0.65, 0.5]
lags =[1,7,14,23,30]

for a in alphas:
    for i in lags:
        data_[f'sales_lag_{i}_alpha_{a}'] = np.log1p(grouped_data['sales'].transform(lambda x: x.shift(i).ewm(alpha=a, min_periods=1).mean()))

data_['sales_lag_7_alpha_0.5'].describe()

In [ ]:
data_['date'] = pd.to_datetime(data_["date"])
data_['day_of_week'] = data_['date'].dt.day_of_week
data_['day_of_year'] = data_['date'].dt.dayofyear
data_['day_of_month'] = data_['date'].dt.day
data_['month'] = data_['date'].dt.month
data_['quarter'] = data_['date'].dt.quarter
data_['year'] = data_['date'].dt.year

# 給料日フラグの作成 (15日と月末)
data_['is_payday'] = (data_['date'].dt.day.isin([15])) | (data_['date'].dt.is_month_end)
data_['is_payday'] = data_['is_payday'].astype(int)

# 国の祝日フラグの作成 (localeがNationalのもの)
data_['is_national_holiday'] = (data_['locale'] == 'National').astype(int)

In [ ]:
data_['onpromotion'] = data_['onpromotion'].apply(lambda x: x > 0)

In [ ]:
sales_lag_columns = list(data_.filter(like="lag").columns)

In [ ]:
# 日付データをdatetime型に変換
data_['date'] = pd.to_datetime(data_['date'])

# 訓練データとテストデータを再定義
data_train = data_[data_['test'] == 0].copy()
data_test = data_[data_['test'] == 1].copy()

# 検証用の期間を学習データの最後の16日間（テスト期間と同じ）に設定
validation_start_date = data_train['date'].max() - pd.DateOffset(days=15)
train_end_date = validation_start_date - pd.DateOffset(days=1)

# 日付に基づいて訓練データと検証データを分割
X_train_df = data_train[data_train['date'] <= train_end_date]
X_val_df = data_train[data_train['date'] >= validation_start_date]

# 特徴量とターゲット（売上）に分ける
features_to_drop = ['test', 'sales', 'id', 'date', 'description', 'transferred']

y_train = X_train_df['sales']
X_train = X_train_df.drop(columns=features_to_drop)

y_val = X_val_df['sales']
X_val = X_val_df.drop(columns=features_to_drop)

X_test = data_test.drop(columns=features_to_drop)

# カテゴリカル変数のカラムを定義
to_dummies = ['day_of_week', 'day_of_month', 'month', 'quarter', 'year', 'store_nbr', 'type_y', 'cluster', 'family', 'onpromotion', 'type_x', 'locale', 'locale_name', 'city', 'state']

# 型を'category'に変換
for col in to_dummies:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
        X_val[col] = X_val[col].astype('category')
        X_test[col] = X_test[col].astype('category')

print(f"訓練データ期間: {X_train_df['date'].min().date()} ~ {X_train_df['date'].max().date()}")
print(f"検証データ期間: {X_val_df['date'].min().date()} ~ {X_val_df['date'].max().date()}")

In [ ]:
X_train = X_train.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x)) 
X_val = X_val.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x)) 
X_test = X_test.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x)) 

In [ ]:
hyper_params = {'task': 'train','boosting_type': 'gbdt','objective': 'regression','metric': ['l1','l2'],'learning_rate': 0.1,
'feature_fraction': 0.9,'bagging_fraction': 0.7,'bagging_freq': 10,'verbose': 0,"max_depth": 50,"num_leaves": 128,"max_bin": 512}

gbm = lgb.LGBMRegressor(**hyper_params)

gbm.fit(X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='l1');

In [ ]:
y_pred = gbm.predict(X_val)

In [ ]:
results = pd.concat([y_val.reset_index(drop=True), pd.Series(y_pred)], axis=1).rename(columns={'sales': 'y_val', 0: 'y_pred'})

results['y_pred'] = results['y_pred'].clip(0)

results = results[results['y_val'] > 10]

results

In [ ]:
y_pred_clipped = results['y_pred'].values

# y_val（実測値）を取得
y_val_values = results['y_val'].values

# RMSLEの計算
rmsle = np.sqrt(mean_squared_log_error(y_val_values, y_pred_clipped))

# 結果の表示
print(f"検証データのRMSLEスコア: {rmsle:.5f}")

In [ ]:
y_pred = gbm.predict(X_test)
y_pred

In [ ]:
data_test['id']
output = pd.DataFrame(index=data_test['id'])
output['sales'] = y_pred
output['sales'] = output['sales'].clip(0)

In [ ]:
output.to_csv('/kaggle/working/submission.csv')